# 从 fMRI 到静息态 ROI 功能连接

本 Notebook 把原脚本拆成可观察的小步骤：读取影像与混杂变量、确认滤波参数、提取并清理 ROI 时间序列、计算 Pearson 相关矩阵并保存结果。运行前请先完成 `00_preprocessing_and_qc.ipynb` 和 `01_inspect_data.ipynb`。

样例只有一名受试者，结果仅用于理解分析流程，不是研究或临床结论。

## 1. 载入功能像与混杂变量

功能像的每个时间点与 confounds 表格的一行对应。头动、白质和脑脊液信号是常见的混杂变量候选，但选择变量时应遵循研究方案。

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nilearn import datasets, image

dataset = datasets.fetch_adhd(n_subjects=1)
func_file = dataset.func[0]
confounds_file = dataset.confounds[0]
func_img = image.load_img(func_file)
confounds = pd.read_csv(confounds_file, sep=None, engine="python")

print(f"影像 shape：{func_img.shape}")
print(f"混杂变量表 shape：{confounds.shape}")
print(f"混杂变量列名：{list(confounds.columns)}")
confounds.head()

**检查点：**混杂变量表行数是否与影像时间点数一致？先看真实列名，再理解下面如何挑选列。自动筛选只是教学示例，正式分析应显式定义并报告混杂变量。

In [ ]:
def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


confound_candidates = {
    "x", "y", "z", "transx", "transy", "transz",
    "motionx", "motiony", "motionz",
    "rotx", "roty", "rotz", "rotationx", "rotationy", "rotationz",
    "motionpitch", "motionroll", "motionyaw", "whitematter", "wm", "csf",
}
selected_columns = [
    column
    for column in confounds.columns
    if normalize_column_name(column) in confound_candidates
]
if not selected_columns:
    raise ValueError(f"未找到熟悉的混杂变量列；现有列：{list(confounds.columns)}")
confounds_selected = confounds[selected_columns].fillna(0)
print(f"本示例选用的混杂变量：{selected_columns}")
print(f"用于回归的矩阵 shape：{confounds_selected.shape}")

缺失值在这里以 0 填充，与保留缺失值或删除时间点相比会改变假设；本示例沿用配套脚本的教学处理。真实研究应检查缺失的原因并预先确定策略。

## 2. 查看一条混杂变量时间序列

下面画出被选中的前几列，横轴是扫描时间点。它帮助我们看到随时间变化的信号，但要量化头动或设定剔除标准，还需要明确指标与研究方案。

In [ ]:
columns_to_plot = selected_columns[:6]
confounds_selected[columns_to_plot].plot(
    subplots=True,
    figsize=(12, 2.5 * len(columns_to_plot)),
    sharex=True,
    title="Selected confounds by time point",
)
plt.xlabel("Time point")
plt.tight_layout()

## 3. 核对时间滤波参数

带通滤波保留指定频率范围内的变化。TR 决定 Nyquist 频率上限 `1 / (2 × TR)`；低通频率必须低于该上限。示例使用 0.01–0.1 Hz，但这不是所有研究都适用的默认标准。

In [ ]:
high_pass = 0.01
low_pass = 0.1
tr = float(func_img.header.get_zooms()[3])
nyquist = 1 / (2 * tr)
print(f"TR：{tr:g} 秒")
print(f"Nyquist 频率：{nyquist:.4f} Hz")
if not (0 < high_pass < low_pass < nyquist):
    raise ValueError(
        f"必须满足 0 < high-pass < low-pass < Nyquist ({nyquist:.4f} Hz)"
    )

## 4. 提取并清理 ROI 时间序列

Harvard–Oxford 皮层图谱给每个体素一个 ROI 标签。`NiftiLabelsMasker` 将每个 ROI 内的体素汇总为一条时间序列，并在这个例子中执行混杂回归、去趋势、滤波和标准化。图谱与功能像需要处于兼容的空间。首次运行还会下载图谱。

In [ ]:
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker

atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
masker = NiftiLabelsMasker(
    labels_img=atlas.maps,
    standardize="zscore_sample",
    standardize_confounds=True,
    detrend=True,
    low_pass=low_pass,
    high_pass=high_pass,
    t_r=tr,
    verbose=1,
)
signals = masker.fit_transform(func_img, confounds=confounds_selected)
roi_names = [str(label) for label in atlas.labels[1:]]
if len(roi_names) != signals.shape[1]:
    raise ValueError("图谱标签数与提取的 ROI 时间序列数不一致。")
print(f"ROI 时间序列 shape（时间点 × ROI）：{signals.shape}")
print(f"ROI 数量：{signals.shape[1]}")
print(f"前几个 ROI 名称：{roi_names[:5]}")

**检查点：**时间序列矩阵的行对应什么，列对应什么？清理前后数据的形状为何通常不变？去噪和滤波会影响后续相关估计，因此解释矩阵时必须报告这些选择。

## 5. 计算并可视化相关矩阵

矩阵中每个元素是两条 ROI 时间序列的 Pearson 相关系数。对角线是 ROI 与自身的相关；矩阵对称。正负值描述样本中的共同变化或相反变化，不说明因果。

In [ ]:
connectivity = np.corrcoef(signals, rowvar=False)
print(f"连接矩阵 shape：{connectivity.shape}")
print(f"矩阵对称：{np.allclose(connectivity, connectivity.T, equal_nan=True)}")
print(f"对角线前几个值：{np.diag(connectivity)[:5]}")

from nilearn import plotting

display = plotting.plot_matrix(
    connectivity,
    labels=roi_names if len(roi_names) == connectivity.shape[0] else None,
    vmin=-1,
    vmax=1,
    colorbar=True,
    title="Resting-state ROI connectivity (Pearson r)",
)

## 6. 保存结果

CSV 保存数值矩阵，PNG 保存可视化，方便在 Notebook 之外查看。输出目录位于当前项目目录下的 `outputs/`。

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
matrix = pd.DataFrame(connectivity, index=roi_names, columns=roi_names)
csv_path = output_dir / "connectivity_matrix.csv"
png_path = output_dir / "connectivity_matrix.png"
matrix.to_csv(csv_path)
display.figure.savefig(png_path, dpi=160, bbox_inches="tight")
plt.close(display.figure)
print(f"已保存：{csv_path.resolve()}")
print(f"已保存：{png_path.resolve()}")

## 回顾与讨论

- 混杂回归、去趋势、滤波、标准化分别想处理什么？哪些选择需要写入方法部分？
- 更换图谱或混杂变量后，矩阵可能怎样变化？为什么不能把单个受试者的相关矩阵当成组水平结论？
- 如果改用不同 TR，滤波器参数检查会如何变化？

相关不等于因果；矩阵也不是独立于预处理策略的客观“真值”。